# Building Blocks of LangGraph

## Tutorials Covered:
1. Creating Simple Agents
2. Memory in LangGraph
3. Tools Integration
4. Error Handling

## 1. Creating Simple Agents

### Learning Objectives:
- Build a basic agent using LangGraph
- Understand agent components and structure
- Implement decision-making capabilities

Agents in LangGraph are specialized graphs that can make decisions about what actions to take next based on their current state and observations.

In [ ]:
# Example of creating a simple agent
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Define the state structure for our agent
class AgentState(TypedDict):
    task: str
    current_step: str
    steps: Annotated[list, operator.add]
    result: str
    agent_decision: str

# Define a simple agent node that decides what to do next
def agent_node(state):
    print(f"Agent processing task: {state['task']}")
    
    # Simple decision logic
    if "calculate" in state['task'].lower():
        decision = "calculator"
        current_step = "Performing calculation"
    elif "summarize" in state['task'].lower():
        decision = "summarizer"
        current_step = "Summarizing content"
    else:
        decision = "general"
        current_step = "Handling general request"
    
    return {
        "current_step": current_step,
        "steps": [current_step],
        "agent_decision": decision
    }

# Define a calculator node
def calculator_node(state):
    task = state['task']
    # Extract numbers and operation from task description
    # This is a simplified example
    if "multiply" in task:
        result = "Multiplication performed"
    elif "add" in task:
        result = "Addition performed"
    else:
        result = "Calculation completed"
    
    return {
        "result": result,
        "steps": ["Calculation completed"]
    }

# Define a summarizer node
def summarizer_node(state):
    return {
        "result": "Content summarized successfully",
        "steps": ["Summary completed"]
    }

# Define a general handler node
def general_node(state):
    return {
        "result": "General request handled",
        "steps": ["General processing completed"]
    }

# Create the agent graph
agent_workflow = StateGraph(AgentState)

agent_workflow.add_node("agent", agent_node)
agent_workflow.add_node("calculator", calculator_node)
agent_workflow.add_node("summarizer", summarizer_node)
agent_workflow.add_node("general", general_node)

agent_workflow.set_entry_point("agent")

# Add conditional edges based on agent decision
def route_agent_decision(state):
    return state.get("agent_decision", "general")

agent_workflow.add_conditional_edges(
    "agent",
    route_agent_decision,
    {
        "calculator": "calculator",
        "summarizer": "summarizer",
        "general": "general"
    }
)

agent_workflow.add_edge("calculator", "__end__")
agent_workflow.add_edge("summarizer", "__end__")
agent_workflow.add_edge("general", "__end__")

agent_app = agent_workflow.compile()

# Test the agent with different tasks
result_calc = agent_app.invoke({
    "task": "Please calculate 15 multiplied by 4",
    "current_step": "",
    "steps": [],
    "result": "",
    "agent_decision": ""
})
print("Agent result for calculation:", result_calc)

result_summary = agent_app.invoke({
    "task": "Please summarize the key points",
    "current_step": "",
    "steps": [],
    "result": "",
    "agent_decision": ""
})
print("Agent result for summary:", result_summary)

## 2. Memory in LangGraph

### Learning Objectives:
- Implement memory mechanisms in LangGraph
- Store and retrieve conversation history
- Manage context across interactions

Memory is crucial for maintaining context in conversational agents and multi-step processes.

In [ ]:
# Example of implementing memory in LangGraph
from datetime import datetime

# Extended state with memory capabilities
class MemoryState(TypedDict):
    user_input: str
    response: str
    conversation_history: Annotated[list, operator.add]
    memory_summary: str
    timestamp: str

# Node that handles user input and maintains memory
def memory_node(state):
    user_input = state['user_input']
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Create a response
    response = f"Received: '{user_input}'. How can I help you further?"
    
    # Update conversation history
    new_message = {
        "timestamp": timestamp,
        "user_input": user_input,
        "response": response
    }
    
    # Simple memory summary update
    current_summary = state.get('memory_summary', '')
    new_summary = f"{current_summary} | {user_input}" if current_summary else user_input
    
    return {
        "response": response,
        "conversation_history": [new_message],
        "memory_summary": new_summary[:200],  # Limit length
        "timestamp": timestamp
    }

# Node that retrieves and uses memory
def recall_node(state):
    history = state.get('conversation_history', [])
    summary = state.get('memory_summary', '')
    
    if history:
        last_interaction = history[-1]
        recall_response = f"I recall you said '{last_interaction['user_input']}' at {last_interaction['timestamp']}. Current summary: {summary}"
    else:
        recall_response = f"No previous interactions. Current summary: {summary}"
    
    return {
        "response": recall_response,
        "conversation_history": [{"timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "recall": recall_response}]
    }

# Create memory-enabled graph
memory_workflow = StateGraph(MemoryState)

memory_workflow.add_node("memory", memory_node)
memory_workflow.add_node("recall", recall_node)

memory_workflow.set_entry_point("memory")
memory_workflow.add_edge("memory", "recall")
memory_workflow.add_edge("recall", "__end__")

memory_app = memory_workflow.compile()

# Test memory functionality
initial_state = {
    "user_input": "Hello, I'm looking for information about LangGraph",
    "response": "",
    "conversation_history": [],
    "memory_summary": "",
    "timestamp": ""
}

memory_result = memory_app.invoke(initial_state)
print("Memory-enabled conversation result:", memory_result)

## 3. Tools Integration

### Learning Objectives:
- Integrate external tools with LangGraph
- Create custom tools for specific tasks
- Handle tool responses in the graph

Tools enable LangGraph agents to interact with external systems and perform complex operations.

In [ ]:
# Example of integrating tools in LangGraph
import json
import requests
from langchain_core.tools import tool

# Define some sample tools
@tool
def search_tool(query: str) -> str:
    """Simulated search tool that returns mock results"""
    # In a real implementation, this would call a search API
    mock_results = f"Mock search results for '{query}': LangGraph is a framework for building stateful, multi-step applications with LLMs."
    return mock_results

@tool
def calculator_tool(expression: str) -> float:
    """Simple calculator tool that evaluates mathematical expressions"""
    try:
        # For security reasons, in production you'd want to use a safer evaluation method
        result = eval(expression)
        return float(result)
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

@tool
def weather_tool(city: str) -> str:
    """Simulated weather tool that returns mock weather data"""
    # In a real implementation, this would call a weather API
    mock_weather = f"Mock weather for {city}: Sunny, 22°C"
    return mock_weather

# State for tool integration
class ToolState(TypedDict):
    query: str
    tool_calls: list
    tool_responses: Annotated[list, operator.add]
    final_response: str

# Node that decides which tool to use
def tool_router(state):
    query = state['query'].lower()
    
    if any(word in query for word in ['calculate', 'math', 'compute']):
        return "calculator"
    elif any(word in query for word in ['weather', 'temperature', 'forecast']):
        return "weather"
    else:
        return "search"

# Tool execution nodes
def execute_search(state):
    query = state['query']
    result = search_tool(query)
    return {
        "tool_responses": [f"Search result: {result}"],
        "final_response": result
    }

def execute_calculator(state):
    # Extract expression from query (simplified)
    query = state['query']
    # This is a simplified extraction - in practice, you'd use more robust parsing
    expression = query.replace('calculate', '').replace('what is', '').strip()
    result = calculator_tool(expression)
    return {
        "tool_responses": [f"Calculator result: {result}"],
        "final_response": f"The result of {expression} is {result}"
    }

def execute_weather(state):
    # Extract city from query (simplified)
    query = state['query']
    # This is a simplified extraction
    city = "New York" if "new york" in query.lower() else query.split()[-1]
    result = weather_tool(city)
    return {
        "tool_responses": [f"Weather result: {result}"],
        "final_response": result
    }

# Create tool-integrated graph
tool_workflow = StateGraph(ToolState)

tool_workflow.add_node("search", execute_search)
tool_workflow.add_node("calculator", execute_calculator)
tool_workflow.add_node("weather", execute_weather)

tool_workflow.set_entry_point("search")  # We'll use conditional routing instead

# Since we can't easily set a router as entry point, we'll create a starter node
def starter_node(state):
    return state

tool_workflow.add_node("starter", starter_node)
tool_workflow.set_entry_point("starter")

tool_workflow.add_conditional_edges(
    "starter",
    tool_router,
    {
        "search": "search",
        "calculator": "calculator",
        "weather": "weather"
    }
)

tool_workflow.add_edge("search", "__end__")
tool_workflow.add_edge("calculator", "__end__")
tool_workflow.add_edge("weather", "__end__")

tool_app = tool_workflow.compile()

# Test tool integration
test_queries = [
    "Tell me about LangGraph",
    "Calculate 15 times 24",
    "What's the weather in London?"
]

for query in test_queries:
    result = tool_app.invoke({"query": query, "tool_calls": [], "tool_responses": [], "final_response": ""})
    print(f"Query: {query}")
    print(f"Result: {result['final_response']}\n")

## 4. Error Handling

### Learning Objectives:
- Implement proper error handling in LangGraph
- Catch and manage exceptions gracefully
- Provide fallback behaviors

Robust error handling is essential for production-ready LangGraph applications.

In [ ]:
# Example of error handling in LangGraph
import random

# State for error handling
class ErrorHandlingState(TypedDict):
    input_data: str
    processing_result: str
    error_occurred: bool
    error_message: str
    retry_count: int
    final_output: str

# Node that might fail randomly (simulating real-world failures)
def risky_operation_node(state):
    input_data = state['input_data']
    retry_count = state.get('retry_count', 0)
    
    # Simulate occasional failure
    if random.random() < 0.3:  # 30% chance of failure
        error_msg = f"Operation failed on attempt {retry_count + 1} with input: {input_data}"
        return {
            "error_occurred": True,
            "error_message": error_msg,
            "retry_count": retry_count + 1
        }
    else:
        # Success case
        result = f"Successfully processed '{input_data}' after {retry_count + 1} attempts"
        return {
            "processing_result": result,
            "error_occurred": False,
            "final_output": result,
            "retry_count": retry_count + 1
        }

# Error handling node
def error_handler_node(state):
    error_msg = state['error_message']
    retry_count = state['retry_count']
    
    print(f"ERROR HANDLED: {error_msg}")
    
    # Decide whether to retry or give up
    if retry_count < 3:  # Retry up to 3 times
        print(f"Retrying... Attempt {retry_count + 1}/3")
        return {"retry_count": retry_count + 1}
    else:
        final_msg = f"Operation permanently failed after {retry_count} attempts. Last error: {error_msg}"
        return {
            "final_output": final_msg,
            "error_occurred": True
        }

# Fallback node for when retries are exhausted
def fallback_node(state):
    input_data = state['input_data']
    fallback_result = f"Using fallback method for '{input_data}'. Manual processing required."
    return {"final_output": fallback_result}

# Create error-handling graph
error_workflow = StateGraph(ErrorHandlingState)

error_workflow.add_node("risky_operation", risky_operation_node)
error_workflow.add_node("error_handler", error_handler_node)
error_workflow.add_node("fallback", fallback_node)

error_workflow.set_entry_point("risky_operation")

# Add conditional edges for error handling
def route_after_risky_operation(state):
    if state.get('error_occurred', False):
        if state.get('retry_count', 0) < 3:
            return "error_handler"
        else:
            return "fallback"
    else:
        return "__end__"

error_workflow.add_conditional_edges(
    "risky_operation",
    route_after_risky_operation,
    {
        "error_handler": "error_handler",
        "fallback": "fallback",
        "__end__": "__end__"
    }
)

# Add loop back from error handler to risky operation
def route_after_error_handling(state):
    if state.get('retry_count', 0) < 3:
        return "risky_operation"  # Retry the operation
    else:
        return "fallback"  # Give up and use fallback

error_workflow.add_conditional_edges(
    "error_handler",
    route_after_error_handling,
    {
        "risky_operation": "risky_operation",
        "fallback": "fallback"
    }
)

error_workflow.add_edge("fallback", "__end__")

error_app = error_workflow.compile()

# Test error handling
print("Testing error handling with possible retries...")
for i in range(3):
    result = error_app.invoke({
        "input_data": f"Test data #{i+1}",
        "processing_result": "",
        "error_occurred": False,
        "error_message": "",
        "retry_count": 0,
        "final_output": ""
    })
    print(f"Run {i+1} result: {result['final_output']}\n")

## Practice Exercises

1. Create an agent that can schedule appointments by integrating with a calendar tool
2. Implement a memory mechanism that summarizes long conversations to prevent context overflow
3. Build a tool that can fetch and process data from a REST API
4. Design an error recovery strategy for a critical business process

## Additional Resources

- [LangGraph Tools Integration Guide](https://langchain-ai.github.io/langgraph/how-tos/tools/)
- [Memory Patterns in LangGraph](https://langchain-ai.github.io/langgraph/how-tos/memory/)
- [Error Handling Best Practices](https://langchain-ai.github.io/langgraph/how-tos/error-handling/)